# OmnisVox CFD — A100 benchmark

**Question:** where does adaptive rollback-refinement (ARR) actually save time on a GPU, and where doesn't it?

For each case it runs a fine **reference**, **uniform grids** at every tier, and **ARR** (with and without rollback), in double and single precision. It then scores every run against the reference and asks the honest question: *is ARR faster than the cheapest uniform grid that is at least as accurate?*

**Before you start**
- Runtime → Change runtime type → **A100 GPU**.
- Put the OmnisVox zip (server or desktop 1.1.1, the engine is the same) anywhere in your Google Drive. The notebook searches for it; if it can't find it, it shows an upload button. Results go to `DRIVE_DIR`.
- Run `MODE = "quick"` first (about 10–20 min). Then `"full"` if the quick results are clean.
- Results are appended to Drive after **every** run, and finished runs are skipped on re-run, so a disconnect only loses the run in progress. Just re-run all cells.

**Scope note:** the experimental *local* (spatial) patch is CPU-only (`local_allow_gpu=False`), so GPU runs here measure ARR's **time-switching** refinement, which is the shipped default.

In [ ]:
# ---- settings ---------------------------------------------------------------
DRIVE_DIR   = "/content/drive/MyDrive/omnisvox"   # results go here (created if missing)
MODE        = "quick"     # "quick" (small grids, ~10-20 min) | "full" (production grids, hours)
PRECISIONS  = ["double", "single"]
REPEATS     = 1           # timed repeats per run (median reported); 3 for publishable timings
DEVICE      = "gpu"       # keep "gpu" on Colab; "cpu" only for a dry run
RUN_CPU_COMPARISON = False  # also time a few small runs on Colab's CPU for a GPU-vs-CPU ratio

In [ ]:
# ---- setup: Drive, package, CuPy ---------------------------------------------
import glob, os, subprocess, sys, zipfile, re, shutil

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

RESULTS_DIR = os.path.join(DRIVE_DIR, f"bench_results_{MODE}")
os.makedirs(RESULTS_DIR, exist_ok=True)      # creates the Drive folder if it doesn't exist

def _find_engine():
    # Find the engine: a zip or an unzipped folder, in DRIVE_DIR, anywhere in Drive, or uploaded.
    roots = [DRIVE_DIR] + (["/content/drive/MyDrive"] if IN_COLAB else []) + ["/content", os.getcwd()]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for depth in ("", "*/", "*/*/", "*/*/*/"):
            hits = glob.glob(os.path.join(root, depth + "*omnisvox*.zip"))
            if hits:
                return "zip", max(hits, key=os.path.getmtime)
            pk = glob.glob(os.path.join(root, depth + "omnisvox_cfd", "__init__.py"))
            if pk:
                return "dir", os.path.dirname(os.path.dirname(pk[0]))
    if IN_COLAB:
        from google.colab import files
        print("Couldn't find the OmnisVox zip in your Drive. Pick it from your computer:")
        up = files.upload()
        name = next((n for n in up if n.lower().endswith(".zip")), None)
        if name:
            return "zip", os.path.abspath(name)
    raise FileNotFoundError("No OmnisVox zip found. Put omnisvox-cfd-...zip in your Drive "
                            f"(e.g. {DRIVE_DIR}) or upload it when asked.")

kind, src = _find_engine()
WORK = "/content/omnisvox" if IN_COLAB else os.path.abspath("omnisvox_work")
if kind == "zip":
    if os.path.isdir(WORK):
        shutil.rmtree(WORK)
    os.makedirs(WORK)
    zipfile.ZipFile(src).extractall(WORK)
    pk = glob.glob(os.path.join(WORK, "**", "omnisvox_cfd", "__init__.py"), recursive=True)
    assert pk, f"{src} does not contain the omnisvox_cfd engine"
    pkg_root = os.path.dirname(os.path.dirname(pk[0]))
else:
    pkg_root = src
sys.path.insert(0, pkg_root)
print("engine from", src, "->", pkg_root)

if DEVICE == "gpu":
    try:
        import cupy  # noqa: F401
    except ImportError:
        smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
        cuda_major = re.search(r"CUDA Version:\s*(\d+)", smi).group(1)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"cupy-cuda{cuda_major}x"], check=True)
        import cupy  # noqa: F401

import omnisvox_cfd
from omnisvox_cfd import Simulation, get_backend, json_safe, gpu_status
from omnisvox_cfd.validation import run_validation, throughput
print("omnisvox_cfd", omnisvox_cfd.__version__)

In [ ]:
# ---- environment record ------------------------------------------------------
import json, platform, time
import numpy as np

env = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "mode": MODE,
       "omnisvox_cfd": omnisvox_cfd.__version__, "python": platform.python_version(),
       "numpy": np.__version__}
if DEVICE == "gpu":
    import cupy as cp
    env["gpu_status"] = gpu_status()
    env["cupy"] = cp.__version__
    env["cuda_runtime"] = cp.cuda.runtime.runtimeGetVersion()
    props = cp.cuda.runtime.getDeviceProperties(0)
    env["gpu_name"] = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    env["gpu_mem_gb"] = round(props["totalGlobalMem"] / 1e9, 1)
    env["nvidia_smi"] = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
json.dump(env, open(os.path.join(RESULTS_DIR, "env.json"), "w"), indent=2)
print({k: v for k, v in env.items() if k != "nvidia_smi"})

## 1. Validate the GPU path
It has never been executed on a GPU before. If anything fails here, stop: the benchmark numbers would mean nothing.

In [ ]:
valid = {}
for prec in PRECISIONS:
    be = get_backend(DEVICE, prec)
    res = run_validation(be)
    valid[prec] = res
    bad = [r["name"] for r in res if not r["passed"]]
    print(f"{prec:7s} {sum(r['passed'] for r in res)}/{len(res)} passed", "" if not bad else f"FAILED: {bad}")
json.dump(json_safe(valid), open(os.path.join(RESULTS_DIR, "validation.json"), "w"), indent=2)
assert all(r["passed"] for rs in valid.values() for r in rs), "Validation failed: do not trust the benchmark."

## 2. Raw throughput
Seconds per RK4 step on uniform grids. This is the ceiling everything else is measured against.

In [ ]:
TP_GRIDS = ([("taylor_green_3d", (n,) * 3) for n in (32, 64, 128, 256)] +
            [("shear_layer", (n, n)) for n in (256, 512, 1024, 2048)])
if MODE == "full":
    TP_GRIDS += [("taylor_green_3d", (384,) * 3), ("shear_layer", (4096, 4096))]
tp = {}
for prec in PRECISIONS:
    be = get_backend(DEVICE, prec)
    throughput(be, TP_GRIDS[:2])                       # warm-up (cuFFT plans)
    rows = throughput(be, TP_GRIDS)
    tp[prec] = rows
    for r in rows:
        print(f"{prec:7s} {'x'.join(map(str, r['shape'])):>14}  {r['seconds_per_step']*1e3:9.2f} ms/step"
              f"  {r['cells_per_second']/1e6:9.1f} Mcells/s")
if RUN_CPU_COMPARISON:
    be = get_backend("cpu", "double")
    tp["cpu_double"] = throughput(be, [g for g in TP_GRIDS if np.prod(g[1]) <= 128**3])
json.dump(json_safe(tp), open(os.path.join(RESULTS_DIR, "throughput.json"), "w"), indent=2)

## 3. Harness
Timings synchronize the GPU before reading the clock, and every grid size is warmed up (cuFFT plan creation) before its first timed run. Accuracy is scored on **statistics** (energy and dissipation histories, peak dissipation), not pointwise fields, because pointwise comparisons of turbulent flows diverge for reasons that have nothing to do with resolution.

In [ ]:
import statistics, traceback

RESULTS = os.path.join(RESULTS_DIR, "runs.jsonl")
_warmed = set()

def load_rows():
    if not os.path.exists(RESULTS):
        return []
    return [json.loads(l) for l in open(RESULTS) if l.strip()]

def done_keys():
    return {r["key"] for r in load_rows() if r.get("status") == "done"}

def save_row(row):
    with open(RESULTS, "a") as f:
        f.write(json.dumps(json_safe(row)) + "\n")

def warm(case_cfg, shapes, be):
    from omnisvox_cfd.cases import get_case
    case = get_case(case_cfg["case"])
    params = case.resolve_params(case_cfg.get("params", {}))
    for s in shapes:
        k = (case.key, tuple(s), be.precision if hasattr(be, "precision") else str(be.describe()))
        if k in _warmed:
            continue
        kw = {"sgs": "none"} if case.ndim == 3 else {}
        sol = case.make_solver(tuple(s), params, be, **kw)
        sol.set_state(case.initial_state(sol, params))
        for _ in range(3):
            sol.step()
        be.synchronize()
        del sol
        be.free_memory()
        _warmed.add(k)

def gpu_peak_reset():
    if DEVICE == "gpu":
        import cupy as cp
        cp.get_default_memory_pool().free_all_blocks()

def gpu_peak():
    if DEVICE == "gpu":
        import cupy as cp
        return cp.get_default_memory_pool().total_bytes() / 1e9
    return None

def run_one(key, cfg, precision, repeats=1, device=None):
    device = device or DEVICE
    be = get_backend(device, precision)
    tiers = cfg["tiers"]
    warm(cfg, tiers, be)
    walls, sim = [], None
    for rep in range(repeats):
        gpu_peak_reset()
        sim = Simulation({**cfg, "device": device, "precision": precision}, backend=be)
        be.synchronize()
        t0 = time.perf_counter()
        sim.run()
        be.synchronize()
        walls.append(time.perf_counter() - t0)
        peak = gpu_peak()
        if rep < repeats - 1:
            del sim
            be.free_memory()
    h = {k: list(map(float, sim.history[k])) for k in ("t", "tier", "energy", "dissipation", "pressure")}
    cost = sim.ctrl.cost_summary()
    t = np.array(h["t"]); tier = np.array(h["tier"])
    dt = np.diff(t, prepend=t[0]) if t.size else t
    time_on_tier = {int(i): float(dt[tier == i].sum() / max(t[-1], 1e-30)) for i in np.unique(tier)} if t.size else {}
    row = {"key": key, "status": sim.status, "error": sim.error, "case": cfg["case"],
           "precision": precision, "device": device, "tiers": tiers, "arr": cfg.get("arr", {}),
           "wall_median": statistics.median(walls), "walls": walls, "steps": sim.ctrl.total_steps,
           "gpu_mem_gb": peak, "rollbacks": cost.get("rollbacks"), "wasted_steps": cost["wasted"]["steps"],
           "cell_step_saving_est": cost.get("cell_step_saving_est"), "time_on_tier": time_on_tier,
           "events": [e["type"] for e in sim.events], "history": h}
    del sim
    be.free_memory()
    return row

def ref_precision(n_cells, ndim):
    # METHOD.md estimate: ~5 GB for 256^3 in double; scale by cells; halve for single.
    if DEVICE != "gpu":
        return "double"
    est_double = 5.0 * n_cells / 256**3 * (1.0 if ndim == 3 else 0.6)
    return "double" if est_double < 0.8 * env["gpu_mem_gb"] else "single"

## 4. Case matrix
| case | what it tests | expectation |
|---|---|---|
| `tgv1600` Taylor–Green, Re 1600 | smooth → transition → decay | ARR should save on the smooth start and the decay |
| `turb3d` decaying 3-D turbulence | busy from the start | roughly break-even |
| `abc` Beltrami ABC flow | laminar control | ARR should stay coarse (best case) |
| `shear2d` 2-D shear layer | roll-up burst | ARR should win |
| `turb2d` 2-D decaying turbulence | cascade, then decay | depends on the run length |

In [ ]:
def uniform_arr(i):
    return {"enabled": False, "start_tier": i}

ARR_ON  = {"enabled": True, "rollback": True}
ARR_NRB = {"enabled": True, "rollback": False}

if MODE == "quick":
    MATRIX = {
        "tgv1600": dict(base={"case": "taylor_green_3d", "params": {"Re": 1600.0}, "t_end": 12.0},
                        tiers=[[32]*3, [64]*3, [128]*3], ref=[256]*3),
        "turb3d":  dict(base={"case": "turbulence_3d", "params": {"Re": 300.0}, "t_end": 4.0},
                        tiers=[[32]*3, [64]*3, [128]*3], ref=[256]*3),
        "abc":     dict(base={"case": "abc_flow", "params": {"Re": 50.0}, "t_end": 5.0},
                        tiers=[[16]*3, [32]*3, [64]*3], ref=[128]*3),
        "shear2d": dict(base={"case": "shear_layer", "params": {"Re": 30000.0, "rho": 100.0}, "t_end": 1.8},
                        tiers=[[256]*2, [512]*2, [1024]*2], ref=[2048]*2),
        "turb2d":  dict(base={"case": "turbulence_2d", "params": {"Re": 1000.0, "k0": 6}, "t_end": 20.0},
                        tiers=[[128]*2, [256]*2, [512]*2], ref=[1024]*2),
    }
else:
    MATRIX = {
        "tgv1600": dict(base={"case": "taylor_green_3d", "params": {"Re": 1600.0}, "t_end": 20.0},
                        tiers=[[64]*3, [128]*3, [256]*3], ref=[512]*3),
        "turb3d":  dict(base={"case": "turbulence_3d", "params": {"Re": 1000.0}, "t_end": 6.0},
                        tiers=[[64]*3, [128]*3, [256]*3], ref=[512]*3),
        "abc":     dict(base={"case": "abc_flow", "params": {"Re": 200.0}, "t_end": 10.0},
                        tiers=[[32]*3, [64]*3, [128]*3], ref=[256]*3),
        "shear2d": dict(base={"case": "shear_layer", "params": {"Re": 30000.0, "rho": 100.0}, "t_end": 1.8},
                        tiers=[[512]*2, [1024]*2, [2048]*2], ref=[4096]*2),
        "turb2d":  dict(base={"case": "turbulence_2d", "params": {"Re": 5000.0, "k0": 8}, "t_end": 20.0},
                        tiers=[[256]*2, [512]*2, [1024]*2], ref=[2048]*2),
    }

def plan():
    jobs = []
    for name, m in MATRIX.items():
        ndim = len(m["ref"])
        rp = ref_precision(int(np.prod(m["ref"])), ndim)
        jobs.append((f"{name}|ref|{rp}", {**m["base"], "tiers": [m["ref"]], "arr": uniform_arr(0)}, rp, 1))
        for prec in PRECISIONS:
            for i, s in enumerate(m["tiers"]):
                jobs.append((f"{name}|uniform{s[0]}|{prec}", {**m["base"], "tiers": m["tiers"], "arr": uniform_arr(i)}, prec, REPEATS))
            jobs.append((f"{name}|arr|{prec}", {**m["base"], "tiers": m["tiers"], "arr": ARR_ON}, prec, REPEATS))
        jobs.append((f"{name}|arr_norollback|double", {**m["base"], "tiers": m["tiers"], "arr": ARR_NRB}, "double", REPEATS))
    return jobs

JOBS = plan()
print(len(JOBS), "runs planned;", len(done_keys()), "already done")

## 5. Run everything
Safe to re-run: finished runs are skipped. A run that fails (e.g. out of memory) is recorded with its error and the loop moves on.

In [ ]:
done = done_keys()
for key, cfg, prec, reps in JOBS:
    if key in done:
        continue
    t0 = time.perf_counter()
    try:
        row = run_one(key, cfg, prec, reps)
    except Exception as exc:
        row = {"key": key, "status": "crashed", "error": f"{type(exc).__name__}: {exc}",
               "trace": traceback.format_exc()[-2000:]}
        get_backend(DEVICE, prec).free_memory()
    save_row(row)
    print(f"{key:36s} {row['status']:8s} {time.perf_counter()-t0:8.1f}s"
          + (f"  ERROR {row['error']}" if row.get("error") else ""), flush=True)

if RUN_CPU_COMPARISON:
    for name in ("abc", "tgv1600"):
        m = MATRIX[name]
        key = f"{name}|arr|cpu_double"
        if key in done_keys():
            continue
        cfg = {**m["base"], "tiers": m["tiers"][:2], "arr": ARR_ON}
        save_row(run_one(key, cfg, "double", 1, device="cpu"))
        print(key, "done", flush=True)

## 6. Results
Errors are relative to the reference run: `diss_err` = relative L2 error of the dissipation history, `peak_err` = relative error of peak dissipation, `energy_err` = relative error of final energy. Each case gets a verdict against the cheapest uniform grid (same precision) whose dissipation error is no worse than ARR's: WIN / TIE / LOSS, TRADE when ARR sits between two grids, and NOISY when finer uniform grids are not more accurate (chaotic late-time behaviour), in which case ARR is compared directly with the finest grid instead.

In [ ]:
import pandas as pd

def metrics(h, ref):
    t, d, e = map(np.asarray, (h["t"], h["dissipation"], h["energy"]))
    rt, rd, re_ = map(np.asarray, (ref["t"], ref["dissipation"], ref["energy"]))
    m = (rt >= max(t[0], rt[0])) & (rt <= min(t[-1], rt[-1]))
    di = np.interp(rt[m], t, d)
    diss_err = float(np.linalg.norm(di - rd[m]) / max(np.linalg.norm(rd[m]), 1e-30))
    peak_err = float(abs(np.nanmax(d) - np.nanmax(rd)) / max(np.nanmax(rd), 1e-30))
    energy_err = float(abs(e[-1] - re_[-1]) / max(abs(re_[-1]), 1e-30))
    return diss_err, peak_err, energy_err

rows = [r for r in load_rows() if r.get("status") == "done"]
refs = {r["key"].split("|")[0]: r for r in rows if r["key"].split("|")[1] == "ref"}
table = []
for r in rows:
    name, kind, prec = r["key"].split("|")
    if kind == "ref" or name not in refs:
        continue
    de, pe, ee = metrics(r["history"], refs[name]["history"])
    table.append({"case": name, "run": kind, "precision": prec, "wall_s": r["wall_median"],
                  "diss_err": de, "peak_err": pe, "energy_err": ee, "steps": r["steps"],
                  "rollbacks": r.get("rollbacks"), "wasted_steps": r.get("wasted_steps"),
                  "gpu_mem_gb": r.get("gpu_mem_gb"), "time_on_tier": r.get("time_on_tier")})
df = pd.DataFrame(table)

verdict = []
for (case, prec), g in df.groupby(["case", "precision"]):
    a = g[g.run == "arr"]
    uni = g[g.run.str.startswith("uniform")]
    if a.empty or uni.empty:
        continue
    a = a.iloc[0]
    finest = uni.loc[uni.wall_s.idxmax()]
    tol = 1.05                      # ARR may be 5% worse and still count as "as accurate"
    # Noise check: if a finer uniform grid is clearly *less* accurate than a coarser one,
    # this metric is not converging (chaotic late-time behaviour), so it cannot rank grids.
    by_res = uni.assign(n=uni.run.str.replace("uniform", "").astype(int)).sort_values("n")
    errs = by_res.diss_err.values
    # (errors below ~1e-3 are at the round-off floor, mostly single precision: not chaos)
    noisy = any(errs[j] > 1.2 * errs[i] and errs[i] > 1e-3
                for i in range(len(errs)) for j in range(i + 1, len(errs)))
    if noisy:
        hs = {r["key"].split("|")[1]: r["history"] for r in rows
              if r["key"].startswith(case + "|") and r["key"].endswith("|" + prec)}
        same = metrics(hs["arr"], hs[finest.run])[0] if ("arr" in hs and finest.run in hs) else float("nan")
        verdict.append({"case": case, "precision": prec, "verdict": "NOISY",
                        "note": (f"metric does not converge with resolution here (finer uniform grids are not "
                                 f"more accurate), so it can't rank grids. ARR vs {finest.run}: dissipation "
                                 f"histories differ by {same:.1e}, ARR {finest.wall_s / a.wall_s:.2f}x faster"),
                        "arr_wall_s": a.wall_s, "arr_diss_err": a.diss_err, "vs_finest": finest.wall_s / a.wall_s})
        continue
    dominators = uni[(uni.wall_s < a.wall_s * 0.95) & (uni.diss_err <= a.diss_err * tol)]
    as_acc = uni[uni.diss_err <= a.diss_err * tol]
    if not dominators.empty:
        d = dominators.loc[dominators.wall_s.idxmin()]
        tag, note = "LOSS", (f"{d.run} is faster ({a.wall_s/d.wall_s:.2f}x) and "
                             f"{a.diss_err/max(d.diss_err,1e-300):.1f}x more accurate")
    elif as_acc.empty:
        tag, note = "WIN", (f"more accurate than every uniform grid "
                            f"({finest.diss_err/max(a.diss_err,1e-300):.2f}x vs {finest.run}) "
                            f"and {finest.wall_s/a.wall_s:.2f}x faster than {finest.run}")
    else:
        c = as_acc.loc[as_acc.wall_s.idxmin()]
        speed = c.wall_s / a.wall_s
        acc_gap = a.diss_err / max(c.diss_err, 1e-300)
        if acc_gap > 3:
            tag = "TRADE"
            note = (f"{speed:.2f}x faster than {c.run}, but {c.run} is {acc_gap:.0f}x more accurate; "
                    f"ARR sits between grids")
        else:
            tag = "WIN" if speed > 1.1 else ("TIE" if speed > 0.9 else "LOSS")
            note = f"{speed:.2f}x vs {c.run} (cheapest uniform as accurate)"
    verdict.append({"case": case, "precision": prec, "verdict": tag, "note": note,
                    "arr_wall_s": a.wall_s, "arr_diss_err": a.diss_err,
                    "vs_finest": finest.wall_s / a.wall_s})
vd = pd.DataFrame(verdict)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
display(df.drop(columns=["time_on_tier"]).sort_values(["case", "precision", "wall_s"]))
display(vd)
df.to_csv(os.path.join(RESULTS_DIR, "runs_table.csv"), index=False)
vd.to_csv(os.path.join(RESULTS_DIR, "verdict.csv"), index=False)

In [ ]:
# Plain-language summary (also saved to Drive)
lines = [f"OmnisVox A100 benchmark ({MODE}), {env.get('gpu_name', DEVICE)}, engine {omnisvox_cfd.__version__}", ""]
for _, v in vd.iterrows():
    lines.append(f"[{v.verdict:5s}] {v.case:8s} {v.precision:6s}: {v.note}")
crashed = [r for r in load_rows() if r.get("status") != "done"]
if crashed:
    lines += ["", "Runs that did not finish:"] + [f"  {r['key']}: {r.get('error')}" for r in crashed]
summary = "\n".join(lines)
print(summary)
open(os.path.join(RESULTS_DIR, "SUMMARY.txt"), "w").write(summary)

In [ ]:
# Accuracy vs wall time, one panel per case (light background, large text)
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 13, "axes.facecolor": "white", "figure.facecolor": "white"})
cases = sorted(df.case.unique())
ncol = min(3, len(cases)); nrow = (len(cases) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(6.0 * ncol, 4.8 * nrow), squeeze=False)
for ax in axes.flat[len(cases):]:
    ax.set_visible(False)
style = {"double": dict(marker="o"), "single": dict(marker="s", fillstyle="none")}
for ax, case in zip(axes.flat, cases):
    g = df[df.case == case]
    for prec, gp in g.groupby("precision"):
        u = gp[gp.run.str.startswith("uniform")].sort_values("wall_s")
        ax.plot(u.wall_s, u.diss_err, color="#1f4e79", lw=2, label=f"uniform ({prec})", **style[prec])
        for _, rr in u.iterrows():
            ax.annotate(rr.run.replace("uniform", ""), (rr.wall_s, rr.diss_err), fontsize=10,
                        xytext=(4, 4), textcoords="offset points", color="#1f4e79")
        a = gp[gp.run == "arr"]
        ax.plot(a.wall_s, a.diss_err, color="#c0392b", ms=11, ls="none", label=f"ARR ({prec})", **style[prec])
        nr = gp[gp.run == "arr_norollback"]
        if not nr.empty:
            ax.plot(nr.wall_s, nr.diss_err, color="#7d3c98", ms=9, ls="none", marker="^",
                    label=f"ARR no rollback ({prec})")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(case); ax.set_xlabel("wall time (s)"); ax.set_ylabel("dissipation error vs reference")
    ax.grid(True, which="both", color="#dddddd")
axes[0][0].legend(fontsize=10, loc="best")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "accuracy_vs_wall.png"), dpi=130)
plt.show()
print("Saved to", RESULTS_DIR)